<a href="https://colab.research.google.com/github/DhrubaAdhikary/Learn_MMIR_with_Dhruv/blob/master/Feature%20Fusion%20early%20and%20late.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Chameleon Early-Fusion Model Colab Demo
# This script demonstrates loading the Chameleon 7B model from Hugging Face
# and shows how a 128x128 image is processed and fused with text.

"""
Prerequisites (Run in a Colab Cell):
!pip install -q transformers accelerate bitsandbytes sentencepiece
!pip install -q flash-attn --no-build-isolation
"""

import torch
from transformers import ChameleonProcessor, ChameleonForConditionalGeneration
from PIL import Image
import requests
from huggingface_hub import login
import io

# Import display from IPython.display for showing images
from IPython.display import display

def run_chameleon_fusion_demo():
    # --- Authentication Step Added Here ---
    # 1. Store your token securely in Google Colab:
    #    Click the "Key" icon (Secrets) on the left sidebar in Colab.
    #    Add a new secret named HF_TOKEN and paste your Hugging Face token.
    #    Toggle "Notebook access" to ON.
    try:
        from google.colab import userdata
        hf_token = userdata.get('HF_TOKEN')
        print("✅ Successfully loaded Hugging Face token from Colab Secrets.")
    except Exception as e:
        print("⚠️ Colab Secret 'HF_TOKEN' not found. Falling back to manual token.")
        hf_token = "YOUR_HF_TOKEN_HERE"

    if hf_token and hf_token != "YOUR_HF_TOKEN_HERE":
        print("Logging into Hugging Face...")
        login(token=hf_token)
    else:
        print("⚠️ WARNING: Hugging Face token not set in script.")
        print("Please replace 'YOUR_HF_TOKEN_HERE' with your actual token or the download will fail.")
        print("-" * 50)

    model_id = "facebook/chameleon-7b" # Note: Requires gated access acceptance on HF

    print("Loading Processor and Model...")
    # Using 4-bit quantization to fit in standard T4/A100 Colab RAM
    processor = ChameleonProcessor.from_pretrained(model_id)
    model = ChameleonForConditionalGeneration.from_pretrained(
        model_id,
        torch_dtype=torch.bfloat16,
        device_map="auto",
        load_in_4bit=True,
        low_cpu_mem_usage=True # Added to prevent Colab system RAM crashes
    )

    # 1. Prepare Image - EXPLICITLY RESIZED TO 128x128
    print("\n--- 1. Fetching and Resizing Image ---")
    # Changed to a reliable HTTPS URL to prevent Colab 'fetch' blocks
    image_url = "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/transformers/tasks/car.jpg"

    # Download with headers to avoid 403 errors, and safely load into PIL
    headers = {"User-Agent": "Mozilla/5.0"}
    response = requests.get(image_url, headers=headers, timeout=15)
    response.raise_for_status() # Ensure the download succeeded

    # Convert to RGB to ensure no alpha-channel issues
    original_image = Image.open(io.BytesIO(response.content)).convert("RGB")
    image_128 = original_image.resize((128, 128))
    print(f"Input image dimensions: {image_128.size}")
    print("Displaying the resized input image:")
    display(image_128) # Display the resized image

    # Note on Architecture:
    # Natively, Chameleon's VQ-VAE uses a 16x16 downsampling factor.
    # Therefore, a true 128x128 image yields (128/16) * (128/16) = 8x8 = 64 visual tokens.
    # Depending on the specific HF processor configuration, it may pad/resize to 512x512,
    # but the underlying architectural math for a 128x128 input is exactly 64 discrete indices.

    # 2. Interleaved Prompt
    # The <image> token tells the processor where to insert the generated visual tokens
    prompt = "What is in this image? <image> Describe the scene briefly."
    print(f"Input Prompt: '{prompt}'")

    # 3. Process and Fuse
    print("\n--- 2. Tokenization and Fusion ---")
    print("The processor converts the text to BPE tokens and the image through a VQ-VAE discretizer.")
    print("These are then interleaved into a single, fused input sequence.")

    inputs = processor(images=image_128, text=prompt, return_tensors="pt").to(model.device, torch.bfloat16)

    # Inspect the flattened fused sequence length
    input_ids = inputs["input_ids"]
    print(f"Fused Sequence Shape (Batch, Sequence Length): {input_ids.shape}")
    print(f"Raw Fused Input IDs (showing the interleaved text and image tokens):\n{input_ids}")
    print("Decoded Fused Input (showing <image_idx> tokens where visual features are inserted):")
    print(processor.decode(input_ids[0], skip_special_tokens=False))

    print("\nThe sequence now contains BOTH text integers and image integers in a single 1D array!")

    # 4. Generate
    print("\n--- 3. Generating Response ---")
    with torch.no_grad():
        generated_ids = model.generate(
            **inputs,
            multimodal_generation_mode="text-only",
            max_new_tokens=50,
            do_sample=True,
            temperature=0.7
        )

    # 5. Decode
    decoded_text = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]
    print("\n--- Model Output ---")
    print(decoded_text)

if __name__ == "__main__":
    try:
        run_chameleon_fusion_demo()
    except Exception as e:
        print(f"Error: {e}")
        print("Note: Ensure you have accepted the license on Hugging Face and are logged in via 'huggingface-cli login'.")

✅ Successfully loaded Hugging Face token from Colab Secrets.
Logging into Hugging Face...
Error: Invalid user token. The token from Google Colab vault is invalid. Please update it from the UI.
Note: Ensure you have accepted the license on Hugging Face and are logged in via 'huggingface-cli login'.


In [ ]:
# CLIP Late-Fusion Model Colab Demo
# This script demonstrates loading the OpenAI CLIP model from Hugging Face
# and shows how images and text are processed independently before "late fusion".

"""
Prerequisites (Run in a Colab Cell):
!pip install -q transformers pillow requests
"""

import torch
import torch.nn.functional as F
from transformers import CLIPProcessor, CLIPModel
from PIL import Image
import requests
import io

def run_clip_late_fusion_demo():
    model_id = "openai/clip-vit-base-patch32"

    print("Loading CLIP Processor and Model...")
    # CLIP does not require gated access, it's open and ready to use
    processor = CLIPProcessor.from_pretrained(model_id)
    model = CLIPModel.from_pretrained(model_id)

    # Move model to GPU if available
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model.to(device)
    print(f"Model loaded on: {device}")

    # 1. Prepare Image
    print("\n--- 1. Fetching Image ---")
    image_url = "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/transformers/tasks/car.jpg"

    headers = {"User-Agent": "Mozilla/5.0"}
    response = requests.get(image_url, headers=headers, timeout=15)
    response.raise_for_status()

    original_image = Image.open(io.BytesIO(response.content)).convert("RGB")
    print(f"Input image loaded successfully.")

    # 2. Prepare Text Prompts
    # Late fusion models excel at zero-shot classification by comparing an image
    # against multiple text descriptions.
    texts = [
        "a photo of a blue car",
        "a photo of a red car",
        "a photo of a cat",
        "a photo of a bustling city street"
    ]
    print(f"\nText Prompts to evaluate:")
    for t in texts:
        print(f" - '{t}'")

    # 3. Independent Encoding (The core of LATE FUSION)
    print("\n--- 2. Independent Modality Encoding ---")
    print("Notice how text and images are encoded through completely separate networks.")

    inputs = processor(text=texts, images=original_image, return_tensors="pt", padding=True).to(device)

    with torch.no_grad():
        # The unified forward pass routes data to the separate Vision and Text encoders under the hood
        outputs = model(**inputs)

        # Pathway A: The Vision Transformer's output (Projected to 512D)
        image_features = outputs.image_embeds

        # Pathway B: The Text Transformer's output (Projected to 512D)
        text_features = outputs.text_embeds

    print(f"Image Features Shape: {image_features.shape}") # (1, 512)
    print(f"Text Features Shape:  {text_features.shape}")  # (4, 512)

    # 4. Late Fusion (The Alignment Space)
    print("\n--- 3. The Late Fusion Point ---")
    print("Fusion happens here: normalizing the separate embeddings and computing their dot product.")

    # Normalize features
    image_features = image_features / image_features.norm(p=2, dim=-1, keepdim=True)
    text_features = text_features / text_features.norm(p=2, dim=-1, keepdim=True)

    # Cosine similarity (The actual "fusion" mathematical operation)
    # logit_scale is a learned temperature parameter in CLIP
    logit_scale = model.logit_scale.exp()
    logits_per_image = logit_scale * image_features @ text_features.t()

    print(f"Similarity Matrix Shape: {logits_per_image.shape}")

    # 5. Output Probabilities
    print("\n--- 4. Final Probabilities ---")
    probs = logits_per_image.softmax(dim=1).cpu().detach().numpy()[0]

    for text, prob in zip(texts, probs):
        print(f"{prob*100:.2f}% : {text}")

if __name__ == "__main__":
    try:
        run_clip_late_fusion_demo()
    except Exception as e:
        print(f"Error during execution: {e}")

Loading CLIP Processor and Model...


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
vision_model.embeddings.position_ids | UNEXPECTED |  | 
text_model.embeddings.position_ids   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded on: cpu

--- 1. Fetching Image ---
Input image loaded successfully.

Text Prompts to evaluate:
 - 'a photo of a blue car'
 - 'a photo of a red car'
 - 'a photo of a cat'
 - 'a photo of a bustling city street'

--- 2. Independent Modality Encoding ---
Notice how text and images are encoded through completely separate networks.
Image Features Shape: torch.Size([1, 512])
Text Features Shape:  torch.Size([4, 512])

--- 3. The Late Fusion Point ---
Fusion happens here: normalizing the separate embeddings and computing their dot product.
Similarity Matrix Shape: torch.Size([1, 4])

--- 4. Final Probabilities ---
97.62% : a photo of a blue car
2.35% : a photo of a red car
0.01% : a photo of a cat
0.01% : a photo of a bustling city street


### Detailed Code Demonstration of CLIP Late-Fusion

Let's break down the `run_clip_late_fusion_demo` function to see the intermediate steps of encoding and similarity calculation more explicitly.

In [ ]:
import torch
import torch.nn.functional as F
from transformers import CLIPProcessor, CLIPModel
from PIL import Image
import requests
import io
import pandas as pd # Added this import

# Re-initialize the model and processor for this demonstration (if not already in memory)
# This assumes the model_id is known and the previous cell ran successfully.
model_id = "openai/clip-vit-base-patch32"

print("Loading CLIP Processor and Model for demonstration...")
processor = CLIPProcessor.from_pretrained(model_id)
model = CLIPModel.from_pretrained(model_id)

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
print(f"Model loaded on: {device}")

# 1. Prepare Image
image_url = "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/transformers/tasks/car.jpg"
headers = {"User-Agent": "Mozilla/5.0"}
response = requests.get(image_url, headers=headers, timeout=15)
response.raise_for_status()
original_image = Image.open(io.BytesIO(response.content)).convert("RGB")
print("Image prepared.")

# 2. Prepare Text Prompts
texts = [
    "a photo of a blue car",
    "a photo of a red car",
    "a photo of a cat",
    "a photo of a bustling city street"
]
print("Text prompts prepared.")

Loading CLIP Processor and Model for demonstration...


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
vision_model.embeddings.position_ids | UNEXPECTED |  | 
text_model.embeddings.position_ids   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded on: cpu
Image prepared.
Text prompts prepared.


#### Step 1: Independent Modality Encoding

The image and text are encoded independently by separate neural networks. The `CLIPProcessor` handles the pre-processing (like resizing images and tokenizing text), and the `CLIPModel`'s internal vision and text encoders produce the feature embeddings.

In [ ]:
# Process inputs through the CLIP processor
inputs = processor(text=texts, images=original_image, return_tensors="pt", padding=True).to(device)

with torch.no_grad():
    outputs = model(**inputs)

    # Extract image and text features (these are the raw embeddings)
    image_features = outputs.image_embeds
    text_features = outputs.text_embeds

print(f"Raw Image Features (first 5 values): {image_features[0, :5]}")
print(f"Image Features Shape: {image_features.shape}") # (Batch_size, Embedding_dim)

print(f"Raw Text Features (first prompt, first 5 values): {text_features[0, :5]}")
print(f"Text Features Shape: {text_features.shape}")  # (Number_of_texts, Embedding_dim)

print("\n--- Visualizing the first text embedding (for 'a photo of a blue car') ---")
# For simplicity, let's just look at the first text embedding
display(pd.DataFrame(text_features[0].cpu().numpy()).head())

Raw Image Features (first 5 values): tensor([ 0.0692, -0.0186, -0.0247, -0.0294,  0.0002])
Image Features Shape: torch.Size([1, 512])
Raw Text Features (first prompt, first 5 values): tensor([ 0.0195, -0.0035, -0.0144,  0.0215, -0.0071])
Text Features Shape: torch.Size([4, 512])

--- Visualizing the first text embedding (for 'a photo of a blue car') ---


,0
0,0.019469
1,-0.003453
2,-0.014358
3,0.021534
4,-0.007121


#### Step 2: Normalization

Before computing similarity, the feature vectors are normalized. This ensures that the magnitude of the vectors doesn't influence the similarity, only their direction.

In [ ]:
# Normalize features to unit vectors
image_features_norm = image_features / image_features.norm(p=2, dim=-1, keepdim=True)
text_features_norm = text_features / text_features.norm(p=2, dim=-1, keepdim=True)

print(f"Normalized Image Features (first 5 values): {image_features_norm[0, :5]}")
print(f"Magnitude of first normalized image feature vector: {torch.norm(image_features_norm[0])}")

print(f"Normalized Text Features (first prompt, first 5 values): {text_features_norm[0, :5]}")
print(f"Magnitude of first normalized text feature vector: {torch.norm(text_features_norm[0])}")

Normalized Image Features (first 5 values): tensor([ 0.0692, -0.0186, -0.0247, -0.0294,  0.0002])
Magnitude of first normalized image feature vector: 1.0
Normalized Text Features (first prompt, first 5 values): tensor([ 0.0195, -0.0035, -0.0144,  0.0215, -0.0071])
Magnitude of first normalized text feature vector: 1.0000001192092896


#### Step 3: Similarity Computation (Dot Product)

Cosine similarity is computed by taking the dot product of the normalized image feature vector with each normalized text feature vector. A `logit_scale` (a learned temperature parameter from CLIP) is applied to scale these similarities.

In [ ]:
# Retrieve the learned logit scale from the model
logit_scale = model.logit_scale.exp()

# Compute cosine similarity using matrix multiplication (dot product)
# image_features_norm (1, 512) @ text_features_norm.t() (512, 4) -> (1, 4)
logits_per_image = logit_scale * (image_features_norm @ text_features_norm.t())

print(f"Similarity Matrix (logits_per_image):\n{logits_per_image}")
print(f"Similarity Matrix Shape: {logits_per_image.shape}")

Similarity Matrix (logits_per_image):
tensor([[28.9423, 25.2175, 20.0062, 20.0676]], grad_fn=<MulBackward0>)
Similarity Matrix Shape: torch.Size([1, 4])


#### Step 4: Convert to Probabilities

Finally, a softmax function is applied to these logits to convert them into a probability distribution, indicating the likelihood of the image matching each text description.

In [ ]:
# Apply softmax to get probabilities
probs = logits_per_image.softmax(dim=1).cpu().detach().numpy()[0]

print("\nFinal Probabilities:")
for i, (text, prob) in enumerate(zip(texts, probs)):
    print(f"{prob*100:.2f}% : {text}")


Final Probabilities:
97.62% : a photo of a blue car
2.35% : a photo of a red car
0.01% : a photo of a cat
0.01% : a photo of a bustling city street


This step-by-step breakdown shows how CLIP, as a late-fusion model, encodes modalities separately, normalizes their embeddings, and then computes similarity between them in a shared embedding space to arrive at the final probabilities.